# US crude stocks: when the visible part and the total move in opposite directions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anefelo/anefelo-notebooks/blob/main/notebooks/2026-11-25-visible-vs-total-crude/visible_vs_total_crude.ipynb)

**Question.** How often does the weekly change in US commercial (ex-SPR) crude stocks point the opposite way from the change in total US crude stocks (commercial plus the Strategic Petroleum Reserve), and how large was the gap in 2022? A tank-shadow tracker sees roughly the first series; a credit or commodity desk cares about the second.

Part of [Anefelo](https://anefelo.substack.com), a newsletter on what satellite data says about the physical economy and its prices.

**Runs in** under a minute on a free Colab CPU. No accounts or keys needed.

**Data:** EIA Weekly Petroleum Status Report series WCESTUS1 (US crude stocks excluding SPR), WCSSTUS1 (SPR) and W_EPC0_SAX_YCUOK_MBBL (Cushing, OK), weekly, thousand barrels. No satellite imagery is used. Full provenance and licences in section 2.

## 0. Setup

In [ ]:
# Colab and GitHub Actions both run this cell. Lower bounds only; exact versions are printed at the end.
%pip install -q "pandas>=2.0" "xlrd>=2.0" "matplotlib>=3.8" "requests>=2.31"

In [ ]:
import datetime as dt
import io
import json
import os
import platform
import random
import sys
from importlib import metadata
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

IN_COLAB = "google.colab" in sys.modules
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

FIG_DIR, RES_DIR = Path("figures"), Path("results")
FIG_DIR.mkdir(exist_ok=True)
RES_DIR.mkdir(exist_ok=True)

plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False})


def get_secret(name):
    """Colab: the Secrets panel (key icon). GitHub Actions: a repository secret of the same name."""
    if IN_COLAB:
        try:
            from google.colab import userdata
            return userdata.get(name)
        except Exception:
            return None
    return os.environ.get(name) or None


SUMMARY = {}


def record(key, value, unit=None, note=None):
    """Every number the post cites is recorded here and written to results/summary.json."""
    if isinstance(value, (np.floating, np.integer)):
        value = value.item()
    SUMMARY[key] = {"value": value, "unit": unit, "note": note}
    return value


def save_figure(fig, name, credit):
    """Saves figures/<name>.png with the data credit printed on the image."""
    fig.text(0.01, -0.01, credit, fontsize=8, color="0.35", ha="left", va="top")
    path = FIG_DIR / f"{name}.png"
    fig.savefig(path, bbox_inches="tight", facecolor="white")
    print("saved", path)
    return path

## 1. Parameters

The period is fixed so a later run gives the same numbers. `END` is the latest EIA week available when this notebook was written (week ending Friday 2026-11-13); move it forward to extend the test.

In [ ]:
NOTEBOOK_NAME = "visible_vs_total_crude.ipynb"
START, END = "2015-01-01", "2026-11-13"   # week-ending dates (EIA weeks end on Friday)
FOCUS_YEAR = 2022                          # the 2022 SPR emergency sale
SERIES = {
    "commercial": "WCESTUS1",               # US ending stocks of crude oil excluding SPR
    "spr": "WCSSTUS1",                      # US ending stocks of crude oil in the SPR
    "cushing": "W_EPC0_SAX_YCUOK_MBBL",     # Cushing, OK ending stocks excluding SPR
}
URL = "https://www.eia.gov/dnav/pet/hist_xls/{code}w.xls"
CREDIT = "Data: US Energy Information Administration (EIA), Weekly Petroleum Status Report; public domain."
INPUTS = {"start": START, "end": END, "focus_year": FOCUS_YEAR, "series": SERIES,
          "urls": {k: URL.format(code=v) for k, v in SERIES.items()}}

## 2. Data and provenance

| Dataset | Provider and access | Series / version | Licence and credit | Citation |
|---|---|---|---|---|
| Weekly U.S. Ending Stocks excluding SPR of Crude Oil | EIA, https://www.eia.gov/dnav/pet/hist_xls/WCESTUS1w.xls | WCESTUS1, weekly, thousand barrels | US government work, public domain; "Data: US Energy Information Administration (EIA), Weekly Petroleum Status Report; public domain." | EIA, Weekly Petroleum Status Report |
| Weekly U.S. Ending Stocks of Crude Oil in SPR | EIA, https://www.eia.gov/dnav/pet/hist_xls/WCSSTUS1w.xls | WCSSTUS1, weekly, thousand barrels | as above | as above |
| Weekly Cushing, OK Ending Stocks excluding SPR of Crude Oil | EIA, https://www.eia.gov/dnav/pet/hist_xls/W_EPC0_SAX_YCUOK_MBBLw.xls | W_EPC0_SAX_YCUOK_MBBL, weekly, thousand barrels | as above | as above |

EIA data are in the public domain and may be reused with attribution (https://www.eia.gov/about/copyrights_reuse.php). Accessed on the date printed in the last cell. The files are revised in place when EIA revises history, so a later download can differ slightly.

## 3. Method

1. Download the three EIA weekly series (thousand barrels, week ending Friday) and join them on the week-ending date.
2. **Total** US crude stocks = commercial (ex-SPR) + SPR. Commercial ex-SPR stands in for the storage a tank-shadow tracker could see; the SPR sits in underground salt caverns that no optical satellite can see.
3. SPR share of total = SPR / total, at the start and end of the period, with its minimum and maximum.
4. Weekly change = this week's level minus last week's. A **sign-disagreement week** is one where the commercial change and the total change are both non-zero and have opposite signs: a tracker of commercial stocks would call a build when total stocks fell, or the reverse. Weeks are counted by their week-ending date, so the first change in the period is the week ending in early January 2015 (versus the last week of 2014).
5. Repeat for calendar 2022, add the cumulative 2022 change in SPR and commercial stocks (last week of 2022 minus last week of 2021), and the correlation of weekly changes, commercial versus total.
6. Context only: the same disagreement count for Cushing, OK (one large tank farm) versus total.

Download each series and check that it parsed into a non-empty weekly table.

In [ ]:
HEADERS = {"User-Agent": "anefelo-notebooks (reproducible research notebook)"}


def load_eia_weekly(code):
    r = requests.get(URL.format(code=code), headers=HEADERS, timeout=60)
    r.raise_for_status()
    raw = pd.read_excel(io.BytesIO(r.content), sheet_name="Data 1", skiprows=2, engine="xlrd")
    raw = raw.rename(columns={raw.columns[0]: "date", raw.columns[1]: code})[["date", code]]
    raw["date"] = pd.to_datetime(raw["date"])
    raw = raw.dropna().set_index("date").sort_index()
    assert len(raw) > 100, f"{code}: only {len(raw)} rows"
    print(f"{code}: {len(raw)} weeks, {raw.index.min().date()} to {raw.index.max().date()}")
    return raw[code].astype(float)


raw = {name: load_eia_weekly(code) for name, code in SERIES.items()}
df = pd.concat(raw, axis=1, join="inner")
assert (df.index.to_series().diff().dropna() == pd.Timedelta(days=7)).mean() > 0.99, "weeks are not regular"
latest = df.index.max()
print("latest week available:", latest.date())
record("latest_week_available_in_download", str(latest.date()), "week-ending date",
       "latest EIA week in the downloaded files at run time; the analysis uses END")

Build total stocks and weekly changes, then cut to the period. Changes are computed before cutting so the first week of the period has a change.

In [ ]:
df["total"] = df["commercial"] + df["spr"]
df["spr_share"] = df["spr"] / df["total"]
chg = df[["commercial", "spr", "total", "cushing"]].diff().add_prefix("d_")
data = df.join(chg)
period = data.loc[START:END].copy()
assert len(period) > 500, "too few weeks in the period"
if latest < pd.Timestamp(END):
    print(f"WARNING: END {END} is after the latest available week {latest.date()}")
first_week, last_week = period.index.min(), period.index.max()
print("period:", first_week.date(), "to", last_week.date(), "|", len(period), "weeks")
record("period_first_week", str(first_week.date()), "week-ending date")
record("period_last_week", str(last_week.date()), "week-ending date", "latest week used")
record("period_weeks", int(len(period)), "weeks", "weekly changes in the period")

(a) The SPR's share of total US crude stocks.

In [ ]:
s = period["spr_share"]
record("spr_share_start", round(s.iloc[0], 4), "share of total US crude stocks (commercial + SPR)", f"week ending {first_week.date()}")
record("spr_share_end", round(s.iloc[-1], 4), "share of total US crude stocks (commercial + SPR)", f"week ending {last_week.date()}")
record("spr_share_min", round(s.min(), 4), "share of total US crude stocks", f"week ending {s.idxmin().date()}")
record("spr_share_min_date", str(s.idxmin().date()), "week-ending date")
record("spr_share_max", round(s.max(), 4), "share of total US crude stocks", f"week ending {s.idxmax().date()}")
record("spr_share_max_date", str(s.idxmax().date()), "week-ending date")
for k in ["commercial", "spr", "total"]:
    record(f"{k}_level_start", int(period[k].iloc[0]), "thousand barrels", f"week ending {first_week.date()}")
    record(f"{k}_level_end", int(period[k].iloc[-1]), "thousand barrels", f"week ending {last_week.date()}")
print(s.describe())

(b), (c), (e) Sign-disagreement weeks and correlation of weekly changes, for the full period and for 2022.

In [ ]:
def disagreement(frame, a="d_commercial", b="d_total"):
    x, y = frame[a], frame[b]
    both = (x != 0) & (y != 0)
    opp = both & (np.sign(x) != np.sign(y))
    return opp, int(both.sum())


def record_block(frame, tag, label):
    opp, n_nonzero = disagreement(frame)
    n = int(len(frame))
    record(f"weeks_{tag}", n, "weeks", label)
    record(f"sign_disagreement_weeks_{tag}", int(opp.sum()), "weeks",
           f"{label}: commercial (ex-SPR) weekly change and total (commercial+SPR) weekly change have opposite signs")
    record(f"sign_disagreement_pct_{tag}", round(100 * opp.sum() / n, 2), "% of weeks", f"{label}; denominator = all weeks")
    record(f"weeks_with_a_zero_change_{tag}", n - n_nonzero, "weeks", "excluded from disagreement (no sign)")
    record(f"corr_weekly_change_commercial_vs_total_{tag}",
           round(frame["d_commercial"].corr(frame["d_total"]), 4), "Pearson r", label)
    # Which way the disagreement ran
    up_c = opp & (frame["d_commercial"] > 0)
    record(f"disagree_commercial_build_total_draw_{tag}", int(up_c.sum()), "weeks",
           "commercial rose while total fell (a visible-stock tracker reads a build)")
    record(f"disagree_commercial_draw_total_build_{tag}", int((opp & ~up_c).sum()), "weeks",
           "commercial fell while total rose (a visible-stock tracker reads a draw)")
    return opp


period["disagree"] = record_block(period, "full", f"{first_week.date()} to {last_week.date()}")
y = period.loc[str(FOCUS_YEAR)].copy()
y["disagree"] = record_block(y, str(FOCUS_YEAR), f"calendar {FOCUS_YEAR} (week-ending dates)")
# Context: disagreement outside 2022
rest = period[period.index.year != FOCUS_YEAR]
opp_rest, _ = disagreement(rest)
record("sign_disagreement_pct_excl_2022", round(100 * opp_rest.sum() / len(rest), 2), "% of weeks",
       "full period excluding calendar 2022")
print({k: v["value"] for k, v in SUMMARY.items() if "disagree" in k or "corr" in k})

(d) Cumulative 2022 change in SPR and commercial stocks: last week of 2022 minus last week of 2021.

In [ ]:
end_prev = data.loc[str(FOCUS_YEAR - 1)].index.max()
end_this = data.loc[str(FOCUS_YEAR)].index.max()
for k in ["spr", "commercial", "total"]:
    delta = data.loc[end_this, k] - data.loc[end_prev, k]
    assert abs(delta - y[f"d_{k}"].sum()) < 1, "cumulative change does not match the sum of weekly changes"
    record(f"cum_change_{FOCUS_YEAR}_{k}", int(delta), "thousand barrels",
           f"level week ending {end_this.date()} minus level week ending {end_prev.date()}")
tot = data.loc[end_this, "total"] - data.loc[end_prev, "total"]
spr = data.loc[end_this, "spr"] - data.loc[end_prev, "spr"]
record(f"spr_share_of_total_change_{FOCUS_YEAR}", round(spr / tot, 3) if tot != 0 else None, "ratio",
       "SPR change / total change over the year; >1 means commercial moved the other way")

Context only: a single-hub tracker (Cushing, OK) versus total US stocks.

In [ ]:
for frame, tag in [(period, "full"), (y, str(FOCUS_YEAR))]:
    opp, _ = disagreement(frame, a="d_cushing", b="d_total")
    record(f"cushing_vs_total_sign_disagreement_pct_{tag}", round(100 * opp.sum() / len(frame), 2), "% of weeks",
           "Cushing weekly change vs total (commercial+SPR) weekly change, opposite signs")
record("cushing_share_of_commercial_end", round(period["cushing"].iloc[-1] / period["commercial"].iloc[-1], 4),
       "share", f"week ending {last_week.date()}")

## 4. Results

Each figure states its takeaway in the title and carries its data credit.

Figure 1: weekly levels of commercial (ex-SPR), SPR and total US crude stocks, with 2022 shaded.

In [ ]:
C_COM, C_SPR, C_TOT = "#0072B2", "#D55E00", "0.15"   # Okabe-Ito blue / vermillion, near-black
fig, ax = plt.subplots(figsize=(10.5, 5.2))
mb = period[["commercial", "spr", "total"]] / 1000
ax.axvspan(pd.Timestamp(f"{FOCUS_YEAR}-01-01"), pd.Timestamp(f"{FOCUS_YEAR}-12-31"), color="0.9", zorder=0)
ax.text(pd.Timestamp(f"{FOCUS_YEAR}-07-01"), mb["total"].max() * 1.01, f"{FOCUS_YEAR}\nSPR emergency sale",
        ha="center", va="bottom", fontsize=9, color="0.3")
ax.plot(mb.index, mb["total"], color=C_TOT, lw=1.8, label="Total (commercial + SPR)")
ax.plot(mb.index, mb["commercial"], color=C_COM, lw=1.6, label="Commercial, excluding SPR (proxy for visible storage)")
ax.plot(mb.index, mb["spr"], color=C_SPR, lw=1.6, label="SPR (underground salt caverns, invisible to tank shadows)")
ax.set_ylim(0, mb["total"].max() * 1.12)
ax.set_ylabel("Million barrels")
ax.xaxis.set_major_locator(mdates.YearLocator(1))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.legend(loc="lower left", frameon=False, fontsize=9)
d22 = SUMMARY[f"cum_change_{FOCUS_YEAR}_spr"]["value"] / 1000
c22 = SUMMARY[f"cum_change_{FOCUS_YEAR}_commercial"]["value"] / 1000
ax.set_title(f"The SPR held {100*SUMMARY['spr_share_start']['value']:.0f}% of US crude stocks in {first_week.year} and "
             f"{100*SUMMARY['spr_share_end']['value']:.0f}% by {last_week.date():%b %Y}\n"
             f"In {FOCUS_YEAR} the SPR fell {abs(d22):.0f} million barrels while commercial stocks changed {c22:+.0f} million",
             loc="left", fontsize=11)
fig.tight_layout()
save_figure(fig, "fig1_us_crude_levels_commercial_spr_total", CREDIT)
plt.show()

Figure 2: 2022 weekly changes, commercial versus total, with sign-disagreement weeks highlighted.

In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 5.2))
w = pd.Timedelta(days=2.6)
for t in y.index[y["disagree"]]:
    ax.axvspan(t - pd.Timedelta(days=3.5), t + pd.Timedelta(days=3.5), color="#F0E442", alpha=0.45, lw=0, zorder=0)
ax.bar(y.index - w / 2, y["d_commercial"] / 1000, width=w, color=C_COM, label="Commercial, excluding SPR")
ax.bar(y.index + w / 2, y["d_total"] / 1000, width=w, color=C_TOT, label="Total (commercial + SPR)")
ax.axhline(0, color="0.3", lw=0.8)
ax.plot([], [], color="#F0E442", alpha=0.6, lw=8, label="Sign-disagreement week")
ax.set_ylabel("Weekly change, million barrels")
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
ax.legend(loc="lower left", frameon=False, fontsize=9, ncol=3)
n22 = SUMMARY[f"sign_disagreement_weeks_{FOCUS_YEAR}"]["value"]
N22 = SUMMARY[f"weeks_{FOCUS_YEAR}"]["value"]
nb = SUMMARY[f"disagree_commercial_build_total_draw_{FOCUS_YEAR}"]["value"]
ax.set_title(f"In {n22} of {N22} weeks of {FOCUS_YEAR}, commercial and total US crude stocks moved in opposite directions\n"
             + (f"In all {nb}, commercial stocks rose while total stocks fell" if nb == n22 else
              f"{nb} of them: commercial stocks rose while total stocks fell"), loc="left", fontsize=11)
fig.tight_layout()
save_figure(fig, f"fig2_{FOCUS_YEAR}_weekly_changes_sign_disagreement", CREDIT)
plt.show()

In [ ]:
summary = {
    "meta": {
        "notebook": NOTEBOOK_NAME,
        "generated_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
        "inputs": INPUTS,
        "units": "levels and changes in thousand barrels unless stated; shares as fractions (0-1); percentages as %",
        "date_range": f"{first_week.date()} to {last_week.date()} (EIA week-ending dates, Fridays)",
        "data_credit": CREDIT,
    },
    "values": SUMMARY,
}
(RES_DIR / "summary.json").write_text(json.dumps(summary, indent=2, default=str))
print(json.dumps(summary, indent=2, default=str))

## 5. What this does not show

- **No satellite product is measured.** No imagery is used. Commercial ex-SPR stocks are a proxy for what a floating-roof-shadow tracker can see, and an imperfect one: per EIA, they include crude in pipelines, at refineries (much of it in fixed-roof tanks, whose fill a shadow cannot read), at tank farms and Alaskan crude in transit by water. So the test bounds the blind-share problem from one side; it does not reproduce any vendor's estimate.
- **Only one kind of invisible storage.** The SPR is the cleanest case because it is reported weekly and lives entirely underground. Floating storage, oil on the water and commercial caverns are not split out here, so the true invisible share is at least the SPR share, not equal to it.
- **Cadence.** EIA reports week-ending Friday levels. A tracker that revisits on other days, or misses weeks to cloud, will see a different set of changes.
- **Attribution.** A sign-disagreement week says the visible and total series moved apart, not why. The notebook does not trace where released SPR barrels went (refinery runs, commercial tanks or exports), only that the visible and total series diverged.
- **Sample.** This is the US, where the reserve is published weekly. Countries whose strategic or commercial stocks sit in undisclosed caverns or tankers cannot be checked this way, which is where the problem is likely largest.
- **Revisions.** EIA revises history in place; numbers here are from the download printed above.

In [ ]:
# Environment, so a reader can match versions if a number differs
packages = ["pandas", "xlrd", "numpy", "matplotlib", "requests"]
print("generated (UTC):", dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"))
print("python:", platform.python_version(), "| colab:", IN_COLAB)
for name in packages:
    try:
        print(f"{name}: {metadata.version(name)}")
    except metadata.PackageNotFoundError:
        print(f"{name}: not installed")